# Text Generation using LSTMs

In [14]:
import os
import sys
import numpy as np
import urllib.request

from keras.models import Sequential
from keras.callbacks import ModelCheckpoint
from tensorflow.keras.utils import to_categorical
from keras.layers import Dense, Dropout, LSTM, Input

In [24]:
MODELS_DIR = 'models'
os.makedirs(MODELS_DIR, exist_ok=True)

## Download the data

The best place to access books that are no longer under Copyright is [Project Gutenberg](https://www.gutenberg.org/). Today we recommend using [Alice’s Adventures in Wonderland by Lewis Carroll](https://www.gutenberg.org/files/11/11-0.txt) for consistency. Of course you can experiment with other books as well.

In [18]:
data_url = 'https://www.gutenberg.org/files/219/219-0.txt'
fname = 'heart_of_darkness.txt'

if fname not in os.listdir():
    urllib.request.urlretrieve(data_url, fname)

## Load data and create character to integer mappings

- Open the text file, read the data then convert it to lowercase letters.
- Map each character to a respective number. Keep 2 dictionaries in order to have more easily access to the mappings both ways around.

In [19]:
# Load data
with open(fname, encoding='utf-8') as f:
    text_data = f.read()
text_data = text_data.lower()

# Characters to integers
chars = sorted(list(set(text_data)))
char_to_int = {char: i for i, char in enumerate(chars)}
int_to_char = {i: char for i, char in enumerate(chars)}

n_chars = len(text_data)
n_vocab = len(chars)

print(f"Total Characters: {n_chars}")
print(f"Total Vocab: {n_vocab}")
print(f"Vocabulary: {char_to_int}")

Total Characters: 209997
Total Vocab: 52
Vocabulary: {'\n': 0, ' ': 1, '!': 2, '&': 3, '(': 4, ')': 5, '*': 6, ',': 7, '-': 8, '.': 9, '0': 10, '1': 11, '2': 12, '6': 13, '9': 14, ':': 15, ';': 16, '?': 17, '[': 18, ']': 19, '_': 20, 'a': 21, 'b': 22, 'c': 23, 'd': 24, 'e': 25, 'f': 26, 'g': 27, 'h': 28, 'i': 29, 'j': 30, 'k': 31, 'l': 32, 'm': 33, 'n': 34, 'o': 35, 'p': 36, 'q': 37, 'r': 38, 's': 39, 't': 40, 'u': 41, 'v': 42, 'w': 43, 'x': 44, 'y': 45, 'z': 46, '—': 47, '‘': 48, '’': 49, '“': 50, '”': 51}


## Prepare the data
- We are "thinking" in sequences of 100 characters: 99 characters in the input and 1 in the output.  
E.g. for the sequence *\['h', 'e', 'l', 'l'\]* as input, we will have *\['o'\]* as the expected output.
- Reshape X such that it has the shape expected by a LSTM: \[samples, time steps, features\].
  - samples: number of data points (len(X));
  - time steps: number of time-dependent steps that are in a single data point (100);
  - features: number of variables for the true value in Y (1).
- Scale the values in X to be in \[0, 1\].
- One-hot encode the true values in Y_modified.

In [20]:
seq_length = 99
dataX = []
dataY = []
for i in range(0, n_chars - seq_length, 1):
    seq_in = text_data[i:i + seq_length]
    seq_out = text_data[i + seq_length]
    dataX.append([char_to_int[char] for char in seq_in])
    dataY.append(char_to_int[seq_out])

n_patterns = len(dataX)
print(f"Total patterns: {n_patterns}")

# Reshape X to be [samples, time steps, features]
X = np.reshape(dataX, (n_patterns, seq_length, 1))
# Normalize
X = X / float(n_vocab)
# One-hot encode the output variable
y = to_categorical(dataY)

print(f"X.shape={X.shape}")
print(f"y.shape={y.shape}")

Total patterns: 209898
X.shape=(209898, 99, 1)
y.shape=(209898, 52)


## Define the LSTM model

- Instantiate the model: a linear stack of layers.
- First layer: LSTM with 256 memory units, input shape from X_new (1st and 2nd). Make sure that this layer returns sequences, such that the next LSTM layer receives sequences and not just random data.
- Second layer: dropout 20% of the neurons of the previous layer in order to avoid overfitting.

******
Optional:
- Third layer: LSTM(256).
- Fourth layer: dropout 20% of the neurons.
******
- Last layer: fully connected with a 'softmax' activation function, and as many neurons as the number of unique characters (the output is one-hot encoded).


Compile the model: categorical_crossentropy, adam.

In [21]:
# Instantiate the model
model = Sequential()

# Add Input layer and first LSTM layer
model.add(Input(shape=(X.shape[1], X.shape[2])))
model.add(LSTM(256, return_sequences=True))

# Add dropout
model.add(Dropout(0.2))

# Add another LSTM layer
model.add(LSTM(256))

# Add dropout
model.add(Dropout(0.2))

# Add a Dense layer
model.add(Dense(y.shape[1], activation='softmax'))

# Compile the model
model.compile(loss='categorical_crossentropy', optimizer='adam')

## Train the model and generate characters

Fit the model for over 100 epochs as the batch size is 30 (ideally). In this case, given the time constraints, we are going to use 5 epochs and a batch size of 128.

Fix a random seed and start generating characters.  The prediction from the model gives out the character encoding of the predicted character, it is then decoded back to the character value and appended to the pattern.  

After enough training time it is going to look like something.

In [25]:
# Load the best weights if they exist (assuming the latest best is 05)
try:
    model.load_weights(os.path.join(MODELS_DIR, "weights-improvement-05-2.2598.keras"))
    print("Loaded best weights from checkpoint.")
except Exception as e:
    print(f"Could not load weights: {e}. Starting training from scratch.")

filepath = os.path.join(MODELS_DIR, "weights-improvement-{epoch:02d}-{loss:.4f}.keras")
checkpoint = ModelCheckpoint(filepath, monitor='loss', verbose=2, save_best_only=True, mode='min')
callbacks_list = [checkpoint]

# fit the model
model.fit(X, y, epochs=5, batch_size=128, callbacks=callbacks_list)

Loaded best weights from checkpoint.
Epoch 1/5
1639/1640 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 2.2007
Epoch 1: loss improved from inf to 2.19165, saving model to models/weights-improvement-01-2.1916.keras
1640/1640 ━━━━━━━━━━━━━━━━━━━━ 52s 32ms/step - loss: 2.2007
Epoch 2/5
1639/1640 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 2.1410
Epoch 2: loss improved from 2.19165 to 2.13743, saving model to models/weights-improvement-02-2.1374.keras
1640/1640 ━━━━━━━━━━━━━━━━━━━━ 50s 30ms/step - loss: 2.1410
Epoch 3/5
1639/1640 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 2.0959
Epoch 3: loss improved from 2.13743 to 2.09164, saving model to models/weights-improvement-03-2.0916.keras
1640/1640 ━━━━━━━━━━━━━━━━━━━━ 50s 31ms/step - loss: 2.0959
Epoch 4/5
1639/1640 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 2.0494
Epoch 4: loss improved from 2.09164 to 2.05068, saving model to models/weights-improvement-04-2.0507.keras
1640/1640 ━━━━━━━━━━━━━━━━━━━━ 50s 31ms/step - loss: 2.0494
Epoch 5/5
1639/1640 ━━━

In [26]:
# pick a random seed
start = np.random.randint(0, len(dataX)-1)
pattern = dataX[start]
print("Random Seed:")
print("\"" + ''.join([int_to_char[value] for value in pattern]) + "\"")

# generate characters
for i in range(100):
    x = np.reshape(pattern, (1, len(pattern), 1))
    x = x / float(n_vocab)
    prediction = model.predict(x, verbose=0)
    index = np.argmax(prediction)
    result = int_to_char[index]
    sys.stdout.write(result)
    pattern.append(index)
    pattern = pattern[1:len(pattern)]

Random Seed:
". i tried to break the spell—the heavy, mute
spell of the wilderness—that seemed to draw him to its"
t a long of the seales of the station of the station of the station of the station of the station of

# Bonus: Words as features

Code here:

https://machinelearningmastery.com/how-to-develop-a-word-level-neural-language-model-in-keras/